# Silver to Gold
## Conversations Notebook - Incremental Load (CDF)
Reads Change Data Feed from the source `conversationtranscript` table starting at the version stored during the initial load, applies the same transformations, and MERGEs changes into the gold target table.

In [0]:
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema  = dbutils.widgets.get("target_schema")
target_table   = dbutils.widgets.get("target_table")
timezone_adjustment = int(dbutils.widgets.get("timezone_adjustment"))

source_table_name = f"{source_catalog}.{source_schema}.conversationtranscript"
target_full = f"{target_catalog}.{target_schema}.{target_table}"

### Step 1: Determine Starting Point
The target table stores the last-processed source version as a table property (`cdf.baseline_version`). We read that version and begin consuming changes from the next version onward.

### Step 2: Transform & Enrich
The following cells apply the same transformations used by the initial load:
1. Filter for `insert` change records
2. Parse the nested JSON `content` column
3. Explode conversation parts (messages)
4. Extract message-level fields with timezone adjustment
5. Join with `systemuser` to resolve sender names

### Step 3: Load to Gold & Advance Checkpoint
Append the transformed records to the gold target table, then update the stored baseline version so the next run picks up only newer changes.

In [0]:
# Retrieve the baseline version stored by the initial load notebook
baseline_row = spark.sql(f"SHOW TBLPROPERTIES {target_full} ('cdf.baseline_version')").collect()

if not baseline_row or baseline_row[0]["value"].startswith("Table"):
    raise Exception(
        f"No 'cdf.baseline_version' property found on {target_full}. "
        "Run the Initial Ingestion notebook first to establish the baseline."
    )

baseline_version = int(baseline_row[0]["value"])
start_version = baseline_version + 1
print(f"Reading CDF from '{source_table_name}' starting at version {start_version}")

### Read Change Data Feed from source
Reads the Delta Change Data Feed from `{source_catalog}.{source_schema}.conversationtranscript` starting at `start_version` (one above the stored baseline). The CDF reader returns all change records along with metadata columns (`_change_type`, `_commit_version`, `_commit_timestamp`).

If no change records are found (count == 0), the notebook exits early via `dbutils.notebook.exit()` to avoid unnecessary processing downstream.

In [0]:
from pyspark.sql.functions import col, array
from delta.tables import DeltaTable

table_name = source_table_name

# Get the current latest version of the table
delta_table = DeltaTable.forName(spark, table_name)
latest_version = (
    delta_table.history(1)
    .select("version")
    .collect()[0][0]
)

print(f"Latest table version: {latest_version}, requested start_version: {start_version}")

# If start_version is beyond the latest committed version, there's nothing new
if start_version > latest_version:
    dbutils.notebook.exit("No new changes to process.")

# Safe to read change data feed now
cdf_df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", start_version)
    .table(source_table_name)
)

change_count = cdf_df.count()
print(f"Found {change_count} change records to process starting from version {start_version}")

if change_count == 0:
    dbutils.notebook.exit("No new changes to process.")

### Filter for insert records and select columns
Filters `cdf_df` to retain only rows where `_change_type == "insert"` — updates and deletes are not expected in this pipeline. Projects the key metadata columns (`id`, `conversation_starttime`, `conversation_startdate`, `bot_conversationtranscriptidname`, `bot_conversationtranscriptId`, `content`) and wraps the `content` string column in a single-element `array()` so the downstream JSON parsing cell can uniformly index it as `content[0]`, matching the initial load structure.

In [0]:
# Filter for insert records only — no updates or deletes expected
upsert_df = cdf_df.filter(
    col("_change_type") == "insert"
).select(
    "id",
    col("conversationstarttime").alias("conversation_starttime"),
    col("conversationstarttime").cast("date").alias("conversation_startdate"),
    "bot_conversationtranscriptidname",
    "bot_conversationtranscriptId",
    "content"
)

# Wrap content in array to match initial load structure
upsert_df = upsert_df.withColumn("content", array(col("content")))

print(f"New inserts: {upsert_df.count()}")

### Parse JSON `content` column and flatten top-level fields
Samples the first element (`content[0]`) of the content array from a non-null row to infer the JSON schema. Because Dataverse exports store the JSON as a double-encoded CSV field — the value is wrapped in outer quotes and inner double-quotes are escaped as `""` — the sample is unwrapped before schema inference: leading and trailing `"` characters are stripped and `""` sequences are replaced with `"`.

The inferred schema is applied via `from_json` to `content[0]` across all rows, producing a `content_json` struct column. That struct's top-level fields are then flattened alongside the key transcript metadata columns (`id`, `conversation_starttime`, `conversation_startdate`, `bot_conversationtranscriptidname`, `bot_conversationtranscriptId`) into `parsed_df`.

If no non-null content values are found the code falls back to returning only the transcript metadata columns.

In [0]:
from pyspark.sql.functions import from_json, schema_of_json

# Infer schema from a sample row
sample_row = (
    upsert_df
    .where(col("content").isNotNull() & (col("content")[0].isNotNull()))
    .select(col("content")[0].alias("content_element"))
    .limit(1)
    .collect()
)

if not sample_row:
    parsed_df = upsert_df.select(
        "id", "conversation_starttime", "conversation_startdate",
        "bot_conversationtranscriptidname", "bot_conversationtranscriptId"
    )
else:
    raw_content = sample_row[0]["content_element"]
    if raw_content.startswith('"') and raw_content.endswith('"'):
        raw_content = raw_content[1:-1].replace('""', '"')

    json_schema = schema_of_json(raw_content)

    ct_with_json = upsert_df.withColumn(
        "content_json",
        from_json(col("content")[0], json_schema)
    )

    parsed_df = ct_with_json.select(
        "id", "conversation_starttime", "conversation_startdate",
        "bot_conversationtranscriptidname", "bot_conversationtranscriptId",
        "content_json.*"
    )

display(parsed_df)

### Detect and explode array field for conversation parts
Automatically detects array-type columns in `parsed_df`, chooses the **first** array column as the conversation-parts container, explodes it to create one row per conversation part, filters to only `type == "message"`, and stores the result as `conversation_df` with a `conversation_part_json` struct column.

If no array-type columns are present in `parsed_df`, the code leaves the data unmodified and assigns `conversation_df = parsed_df`.

In [0]:
from pyspark.sql.types import ArrayType
from pyspark.sql.functions import explode_outer

# Detect and explode array columns (conversation parts)
array_columns = [
    field.name
    for field in parsed_df.schema.fields
    if isinstance(field.dataType, ArrayType)
]

if not array_columns:
    conversation_df = parsed_df
else:
    conversation_array_col = array_columns[0]

    exploded_df = parsed_df.withColumn(
        "conversation_part", explode_outer(col(conversation_array_col))
    )

    filtered_df = exploded_df.where(col("conversation_part.type") == "message")

    conversation_df = filtered_df.select(
        "id", "conversation_starttime", "conversation_startdate",
        "bot_conversationtranscriptidname", "bot_conversationtranscriptId",
        col("conversation_part").alias("conversation_part_json")
    )

display(conversation_df)

### Extract message-level fields from `conversation_part_json`
Projects key fields out of the `conversation_part_json` struct (channel, text, sender details, timestamp), converts the epoch timestamp to a proper Spark `timestamp` with the configured timezone adjustment (`timezone_adjustment` hours), orders messages by newest first, and produces `conversation_df_with_fields` while keeping the original `conversation_part_json` struct for further exploration if needed.

In [0]:
from pyspark.sql.functions import from_unixtime, expr

conversation_df_with_fields = (
    conversation_df
    .select(
        "id",
        expr(f"CAST(conversation_starttime + INTERVAL {timezone_adjustment} HOURS AS TIMESTAMP_NTZ) as conversation_starttime"),
        "conversation_startdate",
        "bot_conversationtranscriptidname",
        "bot_conversationtranscriptId",
        col("conversation_part_json.channelId").alias("channelId"),
        col("conversation_part_json.text").alias("text"),
        col("conversation_part_json.timestamp").cast("double").alias("conversation_part_timestamp"),
        col("conversation_part_json.from.aadObjectId").alias("from_aadObjectId"),
        col("conversation_part_json.from.id").alias("from_id"),
        col("conversation_part_json.from.role").alias("from_role"),
    )
    .withColumn(
        "conversation_part_starttime",
        expr(f"CAST(from_unixtime(conversation_part_timestamp) + INTERVAL {timezone_adjustment} HOURS AS TIMESTAMP_NTZ)")
    )
    .orderBy(col("conversation_part_starttime").desc())
)

display(conversation_df_with_fields)

### Join conversations with user details and derive friendly sender name
Queries the `systemuser` table from `{source_catalog}.{source_schema}` to retrieve user identity information — AAD object ID, full name, and internal email address — into `user_df`. Renames key user columns, left-joins `conversation_df_with_fields` with user data using the AAD object ID (`from_aadObjectId` ⇔ `userentraid`), and constructs a readable `from` field:
- Bot messages (`from_role == 0`) show the bot name (`bot_conversationtranscriptidname`)
- User messages (`from_role == 1`) show the user full name from Dataverse (`userfullname`).

The code also adds a `from_icon` column (🤖 for bot, 👤 for user) and stores the result in `conversation_with_user`.

In [0]:
from pyspark.sql.functions import when, lit

# Load user data
user_df = spark.sql(
    f"SELECT id, azureactivedirectoryobjectid, fullname, internalemailaddress "
    f"FROM {source_catalog}.{source_schema}.systemuser "
    f"WHERE azureactivedirectoryobjectid IS NOT NULL"
)

user_df_renamed = user_df.select(
    col("id").alias("userid"),
    col("azureactivedirectoryobjectid").alias("userentraid"),
    col("fullname").alias("userfullname"),
    col("internalemailaddress").alias("useremailaddress")
)

# Join and derive sender name
conversation_with_user = (
    conversation_df_with_fields.alias("c")
    .join(user_df_renamed.alias("u"), col("c.from_aadObjectId") == col("u.userentraid"), "left")
    .withColumn(
        "from",
        when(col("c.from_role") == 0, col("c.bot_conversationtranscriptidname"))
        .when(col("c.from_role") == 1, col("u.userfullname"))
    )
    .withColumn(
        "from_icon",
        when(col("c.from_role") == 0, lit("🤖"))
        .when(col("c.from_role") == 1, lit("👤"))
        .otherwise(lit(""))
    )
)

display(conversation_with_user)

### Append enriched records to gold target table
Writes the `conversation_with_user` DataFrame to `{target_catalog}.{target_schema}.{target_table}` (defaults: `gold.copilot.conversations`) using Delta `append` mode. Unlike the initial ingestion which overwrites, this incremental load adds only the new records processed in this run.

In [0]:
# Append new records to target table
conversation_with_user.write.format("delta").mode("append").saveAsTable(target_full)

print(f"Appended new records to {target_full}")

### Advance CDF baseline checkpoint
Captures the current (latest) version of the source table via `DESCRIBE HISTORY` and stores it as the `cdf.baseline_version` table property on the target table. The next incremental run will read CDF starting at this version + 1, ensuring no changes are double-processed or missed.

In [0]:
# Update the stored baseline version to the current latest version of the source table
latest_version = (
    spark.sql(f"DESCRIBE HISTORY {source_table_name} LIMIT 1")
    .select("version")
    .collect()[0]["version"]
)

spark.sql(f"""
    ALTER TABLE {target_full}
    SET TBLPROPERTIES ('cdf.baseline_version' = '{latest_version}')
""")

print(f"Updated cdf.baseline_version to {latest_version}")
print(f"Next run will start at version {latest_version + 1}")